# ಪಾಠ 02 - ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೆ임್ವರ್ಕ್ ಅನ್ವೇಷಣೆ

**ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೆ임್ವರ್ಕ್ (MAF)** ಎನ್ನುವುದು AI ಏಜೆಂಟ್‌ಗಳನ್ನು ನಿರ್ಮಿಸಲು ಏಕೀಕೃತ ಫ್ರೆ임್ವರ್ಕ್ ಆಗಿದೆ. ಇದು ನಾಲ್ಕು ಮೂಲ ಕಟ್ಟಡ ಘಟಕಗಳೊಂದಿಗೆ ಸ್ವಚ್ಛ, ಸಂಯೋಜನಾತ್ಮಕ ವಾಸ್ತುಶಿಲ್ಪವನ್ನು ಒದಗಿಸುತ್ತದೆ:

- **ಕ್ಲೈಯಂಟ್** – AI ಮಾದರಿ ಎಂಡ್‌ಪಾಯಿಂಟ್‌ಗೆ ಸಂಪರ್ಕ ಮಾಡುತ್ತದೆ ಮತ್ತು ಸಂವಹನದ ನಿರ್ವಹಣೆ ಮಾಡುತ್ತದೆ
- **ಏಜೆಂಟ್** – ಸೂಚನೆಗಳು ಮತ್ತು ಉಪಕರಣ ವ್ಯಾಖ್ಯಾನಗಳೊಂದಿಗೆ ಕ್ಲೈಯಂಟ್ ಅನ್ನು ಮುಚ್ಚುತ್ತದೆ
- **ಉಪಕರಣಗಳು** – ಮಾದರಿ ಕರೆಮಾಡಬಹುದಾದ ಕಸ್ಟಮ್ ಫಂಕ್ಷನ್‌ಗಳೊಂದಿಗೆ ಏಜೆಂಟ್ ಸಾಮರ್ಥ್ಯಗಳನ್ನು ವಿಸ್ತರಿಸುತ್ತದೆ
- **ಸೆಶನ್** – ಬಹು-ಮಾರ್ಗೀಯ ಸಂವಾದಗಳಿಗಾಗಿ ಸಂವಾದ ಇತಿಹಾಸವನ್ನು ರಕ್ಷಿಸುತ್ತದೆ

ಈ ಪಾಠದಲ್ಲಿ, ನಾವು ಈ ತತ್ತ್ವಗಳನ್ನು ಬಳಸಿಕೊಂಡು ಗಮ್ಯಸ್ಥಳ ಲಭ್ಯತೆಯನ್ನು ಪರಿಶೀಲಿಸುವ **ಪ್ರವಾಸ ಬುಕ್ಕಿಂಗ್ ಏಜೆಂಟ್** ಅನ್ನು ನಿರ್ಮಿಸುವුತೆವೆವು.


## ವ್ಯವಸ್ಥೆ


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್ ವಾಸ್ತುಶಿಲ್ಪವನ್ನು ಅರ್ಥಮಾಡಿಕೊಳ್ಳುವುದು

Microsoft ಎಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್ ಒಂದು ಕ್ರಮವಾರಿಯಾಗಿ ಸ್ತರಬದ್ಧ ವಾಸ್ತುಶಿಲ್ಪವನ್ನು ಅನುಸರಿಸುತ್ತದೆ:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **ಕ್ಲೈಂಟ್** – `FoundryChatClient` ಒಂದು Azure OpenAI ನಿಯೋಜನೆಗೆ ಸಂಪರ್ಕಿಸುತ್ತದೆ. ಇದು ಪ್ರಾಮಾಣೀಕರಣ, ವಿನಂತಿ ಸ್ವರೂಪಗೊಳಿಸುವಿಕೆ ಮತ್ತು ಪ್ರತಿಕ್ರಿಯೆ ವಿಶ್ಲೇಷಣೆಯನ್ನು ನಿರ್ವಹಿಸುತ್ತದೆ.
2. **ಏಜೆಂಟ್** – ಕ್ಲೈಂಟ್ ಮೂಲಕ `provider.create_agent()` ರ ಮೂಲಕ ರಚಿಸಲ್ಪಟ್ಟ ಏಜೆಂಟ್ ಮಾದರಿ ಪ್ರವೇಶವನ್ನು ಸೂಚನೆಗಳೊಂದಿಗೆ (ಸಿಸ್ಟಮ್ ಪ್ರಾಂಪ್ಟ್) ಮತ್ತು ಸಾಧನಗಳೊಂದಿಗೆ ಸಂಯೋಜಿಸುತ್ತದೆ.
3. **ಸಾಧನಗಳು** – Python ಕಾರ್ಯಗಳು `@tool` ಅಲಂಕಾರಗೊಂಡಿದ್ದು, ಏಜೆಂಟ್ ಕ್ರಿಯೆಗಳನ್ನು ಮಾಡಬೇಕಾದ ಅಥವಾ ಮಾಹಿತಿ ಪಡೆಯಬೇಕಾದಾಗ ಅವುಗಳನ್ನು ಕರೆಸಬಹುದು.
4. **ಸೆಶನ್** – ಒಂದು `AgentSession` ವಸ್ತು (`agent.create_session()` ಮೂಲಕ ತಯಾರಾಗುತ್ತದೆ) ಇದು ಸಂಭಾಷಣಾ ಇತಿಹಾಸವನ್ನು ಸಂಗ್ರಹಿಸುತ್ತದೆ, ಹೀಗಾಗಿ ಏಜೆಂಟ್ ಹಿಂದಿನ ಸನ್ನಿವೇಶವನ್ನು ನೆನಪಿಟ್ಟುಕೊಂಡು ಬಹು-ಬದಲಾವಣೆಯ ಸಂವಾದವನ್ನು ಸಾದ್ಯಮಾಡುವುದಕ್ಕೆ ಸಹಾಯ ಮಾಡುತ್ತದೆ.

ಪ್ರತಿಯೊಂದು ಸ್ತರವನ್ನು ಹಂತ ಹಂತವಾಗಿ ನಿರ್ಮಿಸೋಣ.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## @tool ಅಲಂಕರಣದೊಂದಿಗೆ ಉಪಕರಣಗಳನ್ನು ಸೇರಿಸುವುದು

ಉಪಕರಣಗಳು ಏಜೆಂಟ್‌ಗಳಿಗೆ ಪಠ್ಯದ ತಯಾರಿಕೆಯನ್ನು ಮೀರಿದ ಕ್ರಿಯೆಗಳನ್ನ ತೆಗೆದುಕೊಳ್ಳಲು ಅವಕಾಶ ನೀಡುತ್ತವೆ. `@tool` ಅಲಂಕರಣವು ಸಾಮಾನ್ಯ ಪೈಥಾನ್ ಫಂಕ್ಷನ್ ಅನ್ನು ಏಜೆಂಟ್ ಕರೆ ಮಾಡಬಹುದಾದ ಯಾವುದಾದರೂ ವಸ್ತುವಾಗಿ ಪರಿವರ್ತಿಸುತ್ತದೆ.

ಮುಖ್ಯ ಬಿಂದುಗಳು:
- ಪ್ರತಿ ಪರಿಮಾಣವನ್ನು ಮಾದರಿ ಅರ್ಥಮಾಡಿಕೊಳ್ಳಲು `Annotated[type, "description"]` ಅನ್ನು ಬಳಸಿರಿ.
- ಡಾಕ್‌ಸ್ಟ್ರಿಂಗ್ ಆಗಿ ಉಪಕರಣದ ವಿವರಣೆ ಮಾದರಿಗೆ ಕಾಣುತ್ತದೆ.
- `approval_mode="never_require"` ಅಂದರೆ ಉಪಕರಣವು ಬಳಕೆದಾರದ ಅನುಮೋದನೆ ಇಲ್ಲದೆ ಸ್ವಯಂಚಾಲಿತವಾಗಿ ಚಾಲನೆಯಲ್ಲಿ ಇರುತ್ತದೆ.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ಸಾಧನಗಳೊಂದಿಗೆ ಏಜೆಂಟ್ ರಚನೆ

ಈಗ ನಾವು ಕ್ಲೈಂಟ್, ಸೂಚನೆಗಳು ಮತ್ತು ಸಾಧನಗಳನ್ನು ಏಜೆಂಟ್ ಆಗಿ ಸಂಯೋಜಿಸುತ್ತೇವೆ. `ಸೂചനಗಳು` ಸಿಸ್ಟಮ್ ಪ್ರಾಂಪ್ಟ್ನಂತೆ ಕಾರ್ಯನಿರ್ವಹಿಸುತ್ತವೆ — ಅವು ಏಜೆಂಟ್‌ನ ವ್ಯಕ್ತಿತ್ವ ಮತ್ತು ಕಲ್ಯಾಣವನ್ನುนิರ್ಮುತ್ತವೆ.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## ಸೆಷನ್ಗಳೊಂದಿಗೆ ಬಹು-ತಿರುವು ಸಂವರ್ತನೆಗಳು

ಒಂದು `AgentSession` (`agent.create_session()` ಮೂಲಕ ರಚಿಸಲಾಗಿದೆ) ಸಂವರ್ತನೆಯಲ್ಲಿನ ಎಲ್ಲಾ ಸಂದೇಶಗಳನ್ನು ಟ್ರ್ಯಾಕ್ ಮಾಡುತ್ತದೆ. ಪ್ರತಿಯೊಂದು `agent.run()` ಕರೆಗೂ ಅದೇ ಸೆಷನ್ ನೀಡುವುದರಿಂದ, ಏಜೆಂಟ್‌ಗೆ ಸಂಪೂರ್ಣ ಸಂವರ್ತನಾ ಇತಿಹಾಸವನ್ನು ಪ್ರವೇಶಿಸಬಹುದಾಗಿ ಮತ್ತು ಮುಂಚಿನ ಸಂದೇಶಗಳನ್ನು ಉಲ್ಲೇಖಿಸಬಹುದು.

ನಾವು `tools=[check_destination_availability]` ಅನ್ನು ಪೂರೈಸುತ್ತೇವೆ, ಆದ್ದರಿಂದ ಏಜೆಂಟ್ ಪ್ರತಿಯೊಂದು ತಿರುವಿನಲ್ಲಿಯೂ ನಮ್ಮ ಲಭ್ಯತೆ ಪರಿಶೀಲಕವನ್ನು ಕರೆ ಮಾಡಬಹುದು.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಮೈಕ್ರೋಸಾಫ್ಟ್ ಅಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್‌ನ ನಾಲ್ಕು ಸ್ತಂಭಗಳನ್ನು ಅನ್ವೇಷಿಸಿದರು:

| ಕಲ್ಪನೆ | ನೀವು ಕಲಿತದ್ದು |
|---------|------------------|
| **ಗ್ರಾಹಕ** | `FoundryChatClient` ಕರಡು ಪ್ರಮಾಣಪತ್ರ ಆಧಾರಿತ ಪ್ರಾಮಾಣೀಕರಣದೊಂದಿಗೆ Azure OpenAIಗೆ ಸಂಪರ್ಕಿಸುತ್ತದೆ |
| **ಅಜೆಂಟ್** | `provider.create_agent()` ಮಾದರಿ ಸಂಪರ್ಕವನ್ನು ನಿರ್ದೇಶಗಳು ಮತ್ತು ಹೆಸರಿನೊಂದಿಗೆ ಗುಂಪುಮಾಡುತ್ತದೆ |
| **ಪರಿಕರಗಳು** | `@tool` ಸಜ್ಜೆ Python ಕಾರ್ಯಗಳನ್ನು ಅಜೆಂಟ್ ಕರೆ ಮಾಡಲು ಅನಾವರಣಗೊಳಿಸುತ್ತದೆ |
| **ಸತ್ರ** | `agent.create_session()` ಬಹುಮಟ್ಟದ ಸಂಭಾಷಣೆಯ ಇತಿಹಾಸವನ್ನು ಕಾಯ್ದುಕೊಳ್ಳುತ್ತದೆ |

ಈ ನಿರ್ಮಾಣ ಘಟಕಗಳು ಒಟ್ಟಿಗೆ ಸೇರಿ ನೈಸರ್ಗಿಕ ಸಂಭಾಷಣೆಗಳನ್ನು ಹಿಡಿದುಕೊಳ್ಳುವ, ಬಾಹ್ಯ ಕಾರ್ಯಗಳನ್ನು ಕಾಲ್ ಮಾಡುವ, ಮತ್ತು ಸಂಧರ್ಭವನ್ನು ಕಾಯ್ದುಕೊಳ್ಳುವ ಅಜೆಂಟ್ ಗಳನ್ನು ರಚಿಸುತ್ತವೆ — ಮುಂದಿನ ಪಾಠಗಳಲ್ಲಿ ಹೆಚ್ಚಿನ ಸುಧಾರಿತ ಏಜೆಂಟಿಕ್ ಮಾದರಿಗಳಿಗಾಗಿ ಆಧಾರದ ಕೋಶವಾಗಿದೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
